# Day 6 — Practice Questions: API Response Handling

> **Real APIs used (free, no key required):**
> - **JSONPlaceholder** — https://jsonplaceholder.typicode.com
> - **Open-Meteo** — https://api.open-meteo.com
> - **REST Countries** — https://restcountries.com
>
> **Topics tested:** Pagination · Throttling · Retry + Backoff · Flatten nested JSON · Error isolation

Run the setup cell first, then attempt each question before revealing the solution.

In [ ]:
import requests
import time
from pprint import pprint

# get_with_retry is defined in api_helpers.py — open it to see the full retry logic
from api_helpers import get_with_retry

print('Setup complete. get_with_retry() is ready.')

---
## Q1 (Easy) — Fetch and Flatten Paginated Posts

**API:** `GET https://jsonplaceholder.typicode.com/posts?_page=N&_limit=10`  
There are **100 posts** total across 10 pages.

Write `fetch_all_posts(page_size, delay)` that:
1. Loops through all pages using `_page` / `_limit` query params
2. Stops when an empty page is returned
3. Sleeps `delay` seconds between each page (throttling)
4. Uses `get_with_retry()` for each request
5. Returns a flat list of all post dicts

**Expected:** 100 posts total, fetched in pages of 10.

**Warm-ups:**
- What query params does JSONPlaceholder use for pagination?
- What does an empty list `[]` as a response mean?
- Why use `list.extend()` instead of `list.append()` when adding a page?

In [ ]:
def fetch_all_posts(page_size=10, delay=0.3):
    """
    Fetch all posts from JSONPlaceholder with pagination + throttling.
    Returns flat list of post dicts.
    """
    pass

In [ ]:
# --- assert ---
posts = fetch_all_posts(page_size=10, delay=0.3)

assert posts is not None,       'Function returned None'
assert isinstance(posts, list), 'Return type must be list'
assert len(posts) == 100,       f'Expected 100 posts, got {len(posts)}'

# Every record must have the standard JSONPlaceholder post fields
required_keys = {'userId', 'id', 'title', 'body'}
for p in posts:
    assert required_keys.issubset(p.keys()), f'Post missing keys: {p.keys()}'

# IDs must be unique and range 1-100
ids = [p['id'] for p in posts]
assert len(set(ids)) == 100,     f'Duplicate post IDs found'
assert min(ids) == 1 and max(ids) == 100

print(f'Q1 passed! Fetched {len(posts)} posts.')
print(f'  Users represented: {sorted(set(p["userId"] for p in posts))}')
print(f'  First post title: "{posts[0]["title"]}"')

### Solution

In [ ]:
# SOLUTION
def fetch_all_posts(page_size=10, delay=0.3):
    all_posts, page = [], 1
    while True:
        batch = get_with_retry(
            'https://jsonplaceholder.typicode.com/posts',
            params={'_page': page, '_limit': page_size},
        )
        if not batch:          # empty page → no more data
            break
        all_posts.extend(batch)
        print(f'  Page {page:2d}: +{len(batch)} posts (total: {len(all_posts)})')
        page += 1
        time.sleep(delay)      # throttle between pages
    return all_posts

posts = fetch_all_posts(page_size=10, delay=0.3)
print(f'\nTotal posts: {len(posts)}')

---
## Q2 (Medium) — Fetch Weather for Multiple Cities with Retry + Backoff

**API:** `GET https://api.open-meteo.com/v1/forecast`  
No API key needed. Parameters: `latitude`, `longitude`, `current`, `timezone`.

Write `fetch_weather_batch(cities, delay)` that:
1. Loops through a list of city dicts (`name`, `lat`, `lon`)
2. Calls `get_with_retry()` for each city (retry on 429/5xx, raise on 4xx)
3. Sleeps `delay` seconds between cities (throttling)
4. Flattens the response: extracts `temperature_2m`, `wind_speed_10m`, `relative_humidity_2m` from `data['current']`
5. If a city fetch fails after all retries, records it in a `failed` list with the city name and error message — does NOT crash the whole batch
6. Returns `(records, failed)` — two lists

**API response shape:**
```json
{
  "current": {
    "time": "2024-01-15T14:00",
    "temperature_2m": 22.5,
    "wind_speed_10m": 12.3,
    "relative_humidity_2m": 65
  }
}
```

**Warm-ups:**
- Where in the response is the current temperature? (`data['current']['temperature_2m']`)
- What `try/except` exception should you catch if `get_with_retry()` exhausts all retries?
- Why return `(records, failed)` instead of just `records`?

In [ ]:
CITIES = [
    {'name': 'Delhi',     'lat': 28.6139,  'lon': 77.2090},
    {'name': 'Mumbai',    'lat': 19.0760,  'lon': 72.8777},
    {'name': 'London',    'lat': 51.5074,  'lon': -0.1278},
    {'name': 'New York',  'lat': 40.7128,  'lon': -74.0060},
    {'name': 'Tokyo',     'lat': 35.6762,  'lon': 139.6503},
    {'name': 'Sydney',    'lat': -33.8688, 'lon': 151.2093},
]

def fetch_weather_batch(cities, delay=0.5):
    """
    Fetch current weather for each city using Open-Meteo API.
    Returns (records, failed).
    records: list of flat dicts with city weather data
    failed:  list of {city, error} for cities that couldn't be fetched
    """
    pass

In [ ]:
# --- assert ---
records, failed = fetch_weather_batch(CITIES, delay=0.5)

assert isinstance(records, list), 'records must be a list'
assert isinstance(failed,  list), 'failed must be a list'
assert len(records) + len(failed) == len(CITIES), \
    f'Total should equal {len(CITIES)}, got records={len(records)} failed={len(failed)}'

# Every successful record must have required fields
required = {'city', 'temp_c', 'wind_kmh', 'humidity_pct', 'fetched_at'}
for r in records:
    assert required.issubset(r.keys()), f'Record missing fields: {r.keys()}'
    assert isinstance(r['temp_c'], (int, float)),   f"temp_c not a number: {r['temp_c']}"
    assert isinstance(r['wind_kmh'], (int, float)), f"wind_kmh not a number: {r['wind_kmh']}"

# Every failed entry must have city name and error
for f in failed:
    assert 'city'  in f, f'failed entry missing city: {f}'
    assert 'error' in f, f'failed entry missing error: {f}'

print(f'Q2 passed! {len(records)} cities fetched, {len(failed)} failed.')
print('\nWeather report:')
for r in records:
    print(f"  {r['city']:10s}: {r['temp_c']:5.1f}°C  wind {r['wind_kmh']:5.1f} km/h  humidity {r['humidity_pct']}%")
if failed:
    print('\nFailed:')
    for f in failed:
        print(f"  {f['city']}: {f['error']}")

### Solution

In [ ]:
# SOLUTION
WEATHER_URL = 'https://api.open-meteo.com/v1/forecast'

def fetch_weather_batch(cities, delay=0.5):
    records, failed = [], []

    for city in cities:
        try:
            data = get_with_retry(
                WEATHER_URL,
                params={
                    'latitude':  city['lat'],
                    'longitude': city['lon'],
                    'current':   [
                        'temperature_2m',
                        'wind_speed_10m',
                        'relative_humidity_2m',
                    ],
                    'timezone':  'auto',
                    'forecast_days': 1,
                },
            )
            current = data['current']
            records.append({
                'city':         city['name'],
                'temp_c':       current.get('temperature_2m'),
                'wind_kmh':     current.get('wind_speed_10m'),
                'humidity_pct': current.get('relative_humidity_2m'),
                'fetched_at':   current.get('time'),
            })
            print(f"  {city['name']:10s}: OK")

        except Exception as e:
            # Do NOT crash the batch — record and continue
            failed.append({'city': city['name'], 'error': str(e)})
            print(f"  {city['name']:10s}: FAILED — {e}")

        time.sleep(delay)  # throttle

    return records, failed

records, failed = fetch_weather_batch(CITIES, delay=0.5)
print(f'\nFetched: {len(records)}  Failed: {len(failed)}')
for r in records:
    print(f"  {r['city']:10s}: {r['temp_c']}°C")

---
## Q3 (Medium) — Enrich Posts with User Data + Flatten Deep Nesting

**APIs used:**
- `GET https://jsonplaceholder.typicode.com/posts?_page=N&_limit=10` — paginated posts
- `GET https://jsonplaceholder.typicode.com/users/{userId}` — user lookup

Write `fetch_enriched_posts(num_pages, page_size, delay)` that:
1. Fetches `num_pages` pages of posts (paginated, with throttle `delay` between pages)
2. For each post, fetches the author's user record — **cache users** so the same user is only fetched once
3. Flattens everything into one dict per post with keys: `post_id`, `title`, `user_id`, `user_name`, `user_email`, `user_company`, `user_city`
4. `user_company` lives at `user['company']['name']` — use `.get()` safely
5. `user_city` lives at `user['address']['city']`
6. Throttle `delay` seconds between user fetches too (only for cache misses)
7. Returns the flat list of enriched post dicts

**Expected for 2 pages (20 posts):** 20 flat dicts, each with all 7 keys.  
JSONPlaceholder has 10 users total — your cache should have at most 10 entries.

**Warm-ups:**
- How do you access `user['company']['name']` safely when `company` might be missing?
- What data structure is best for a user cache keyed by `user_id`?
- How do you check if a user_id is already in the cache before making an API call?

In [ ]:
def fetch_enriched_posts(num_pages=2, page_size=10, delay=0.3):
    """
    Fetch paginated posts and enrich each with author info from the users endpoint.
    Cache users to avoid redundant API calls.
    Returns list of flat dicts.
    """
    pass

In [ ]:
# --- assert ---
enriched = fetch_enriched_posts(num_pages=2, page_size=10, delay=0.3)

assert enriched is not None,       'Function returned None'
assert isinstance(enriched, list), 'Return type must be list'
assert len(enriched) == 20,        f'Expected 20 posts (2 pages × 10), got {len(enriched)}'

# All 7 required keys present on every record
required = {'post_id', 'title', 'user_id', 'user_name', 'user_email', 'user_company', 'user_city'}
for r in enriched:
    assert required.issubset(r.keys()), f'Missing keys: {required - set(r.keys())} in {r}'

# user_name and user_email must be non-empty strings
for r in enriched:
    assert isinstance(r['user_name'],  str) and r['user_name'],  f"Empty user_name in: {r}"
    assert isinstance(r['user_email'], str) and r['user_email'], f"Empty user_email in: {r}"

# post_ids should be unique (posts 1-20)
post_ids = [r['post_id'] for r in enriched]
assert len(set(post_ids)) == 20, 'Duplicate post_ids'
assert min(post_ids) == 1 and max(post_ids) == 20

print(f'Q3 passed! {len(enriched)} enriched posts.')
print('\nSample records:')
for r in enriched[:3]:
    print(f"  post {r['post_id']:3d} | {r['user_name']:20s} | {r['user_company']:25s} | {r['user_city']}")

### Solution

In [ ]:
# SOLUTION
POSTS_URL = 'https://jsonplaceholder.typicode.com/posts'
USERS_URL = 'https://jsonplaceholder.typicode.com/users'

def fetch_enriched_posts(num_pages=2, page_size=10, delay=0.3):
    user_cache = {}    # {user_id: user_dict} — fetch each user at most once
    enriched   = []

    for page in range(1, num_pages + 1):
        # ── Fetch one page of posts ──────────────────────────────────────────
        posts = get_with_retry(
            POSTS_URL,
            params={'_page': page, '_limit': page_size},
        )
        print(f'Page {page}: fetched {len(posts)} posts')
        time.sleep(delay)   # throttle between pages

        for post in posts:
            uid = post['userId']

            # ── Fetch user only if not already cached ───────────────────────
            if uid not in user_cache:
                user_cache[uid] = get_with_retry(f'{USERS_URL}/{uid}')
                time.sleep(delay)   # throttle user lookups too

            user = user_cache[uid]

            enriched.append({
                'post_id':      post['id'],
                'title':        post['title'],
                'user_id':      uid,
                'user_name':    user.get('name', 'N/A'),
                'user_email':   user.get('email', 'N/A'),
                # Nested safe access — company and address may be absent
                'user_company': user.get('company', {}).get('name', 'N/A'),
                'user_city':    user.get('address', {}).get('city', 'N/A'),
            })

    print(f'\nUser cache size: {len(user_cache)} unique users fetched')
    return enriched

enriched = fetch_enriched_posts(num_pages=2, page_size=10, delay=0.3)
print(f'Enriched posts: {len(enriched)}')
print('\nSample:')
for r in enriched[:3]:
    print(f"  post {r['post_id']:3d} | {r['user_name']:20s} | {r['user_company']:25s} | {r['user_city']}")

---
## Bonus — REST Countries: Flatten + Filter

Not graded — explore on your own.  
Fetch all Asian countries, flatten deeply nested JSON, sort by population.

In [ ]:
resp = requests.get('https://restcountries.com/v3.1/region/asia', timeout=15)
resp.raise_for_status()
countries = resp.json()

def flatten_country(c):
    currencies = c.get('currencies', {})
    currency   = list(currencies.keys())[0] if currencies else 'N/A'
    languages  = c.get('languages', {})
    return {
        'name':       c['name']['common'],
        'capital':    c.get('capital', ['N/A'])[0] if c.get('capital') else 'N/A',
        'population': c.get('population', 0),
        'area_km2':   c.get('area', 0),
        'currency':   currency,
        'languages':  ', '.join(languages.values()) if languages else 'N/A',
        'subregion':  c.get('subregion', 'N/A'),
    }

flat = sorted([flatten_country(c) for c in countries],
              key=lambda x: x['population'], reverse=True)

print(f'{len(flat)} Asian countries. Top 10 by population:')
for c in flat[:10]:
    print(f"  {c['name']:20s}  {c['population']:>12,}  {c['capital']:15s}  {c['subregion']}")